# Weight Initialization

## Learning Objectives
1. Compare Xavier, He, and random initialization — measure activation variance across layers
2. Train with different init schemes in PyTorch; compare convergence speed
3. Diagnose dead neurons from zero-init and fix with proper initialization
4. Visualise gradient flow per layer for Xavier vs He vs constant init


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — Activation Variance: Xavier vs He vs Random

In [ ]:
# ── Level 1: activation variance across layers (numpy) ───────────────────
import numpy as np

np.random.seed(42)
N_LAYERS = 8
FAN_IN   = 256   # units per layer

def forward_activations(init_scale, activation="tanh"):
    # Propagate a random input through N_LAYERS and record activation variance.
    x = np.random.randn(1000, FAN_IN)
    variances = [float(x.var())]
    for _ in range(N_LAYERS):
        W = np.random.randn(FAN_IN, FAN_IN) * init_scale
        x = W @ x.T   # shape: (FAN_IN, 1000)
        if activation == "tanh":
            x = np.tanh(x)
        elif activation == "relu":
            x = np.maximum(0, x)
        x = x.T        # back to (1000, FAN_IN)
        variances.append(float(x.var()))
    return variances

# Xavier scale = 1/sqrt(fan_in)  — designed for tanh/sigmoid activations
# He scale     = sqrt(2/fan_in)  — designed for ReLU activations
# Large scale  = 1.0             — leads to exploding/saturating
# Small scale  = 0.01            — leads to vanishing

inits = {
    "Xavier (tanh)": (1 / np.sqrt(FAN_IN), "tanh"),
    "He (relu)":     (np.sqrt(2 / FAN_IN), "relu"),
    "Too large":     (1.0,                  "tanh"),
    "Too small":     (0.01,                 "tanh"),
}

print(f"{'Init':<20}  {'Layer 0':>8}", "  ".join(f"L{i:1d}" for i in range(1, N_LAYERS + 1)))
for name, (scale, act) in inits.items():
    variances = forward_activations(scale, act)
    row = f"{name:<20}  {variances[0]:>8.4f}"
    for v in variances[1:]:
        row += f"  {v:5.4f}"
    print(row)

print("\nXavier/He maintain stable variance; large scale → explode; small → vanish.")


## Level 2 — PyTorch Init Schemes and Loss Curve Comparison

In [ ]:
# ── Level 2: compare init schemes on a real training task ────────────────
import torch, copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

X = torch.randn(600, 64)
y = (X[:, :4].sum(1) > 0).long()
loader = DataLoader(TensorDataset(X[:500], y[:500]), batch_size=32, shuffle=True)
X_v, y_v = X[500:].to(device), y[500:].to(device)

def make_deep_net():
    return nn.Sequential(
        nn.Linear(64, 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 64),  nn.ReLU(),
        nn.Linear(64, 2)
    )

def apply_init(model, scheme):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if scheme == "xavier":
                nn.init.xavier_uniform_(m.weight)
            elif scheme == "kaiming":
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif scheme == "constant_01":
                nn.init.constant_(m.weight, 0.01)
            elif scheme == "large_random":
                nn.init.normal_(m.weight, std=1.5)
            nn.init.zeros_(m.bias)
    return model

def train_and_track(scheme, epochs=30):
    m   = apply_init(make_deep_net(), scheme).to(device)
    opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    losses = []
    for _ in range(epochs):
        m.train()
        ep_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            l = F.cross_entropy(m(xb), yb)
            l.backward()
            opt.step()
            ep_loss += l.item()
        losses.append(ep_loss / len(loader))
    m.eval()
    with torch.no_grad():
        acc = (m(X_v).argmax(1) == y_v).float().mean().item()
    return losses, acc

schemes = ["xavier", "kaiming", "constant_01", "large_random"]
print(f"{'Scheme':<16}  {'Epoch 1 Loss':>13}  {'Final Loss':>11}  {'Val Acc':>8}")
print("-" * 55)
for sch in schemes:
    losses, acc = train_and_track(sch)
    print(f"{sch:<16}  {losses[0]:>13.4f}  {losses[-1]:>11.4f}  {acc:>8.3f}")


## Real-World Example 1 — Dead Neurons from Zero Init

In [ ]:
# ── RW1: diagnose dead neurons caused by bad initialisation ───────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

X = torch.randn(400, 32)
y = (X[:, 0] + X[:, 2] > 0).long()
loader = DataLoader(TensorDataset(X[:300], y[:300]), batch_size=32, shuffle=True)
X_v, y_v = X[300:].to(device), y[300:].to(device)

class NetWithStats(nn.Module):
    # Network that records per-layer activation statistics.
    def __init__(self, init_scheme):
        super().__init__()
        self.fc1 = nn.Linear(32, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 2)
        self._init(init_scheme)
        self.activation_stats = {}

    def _init(self, scheme):
        for fc in [self.fc1, self.fc2, self.fc3]:
            if scheme == "zeros":
                nn.init.zeros_(fc.weight)
                nn.init.zeros_(fc.bias)
            elif scheme == "kaiming":
                nn.init.kaiming_normal_(fc.weight, nonlinearity="relu")
                nn.init.zeros_(fc.bias)

    def forward(self, x):
        for i, (layer, name) in enumerate([(self.fc1, "fc1"), (self.fc2, "fc2")]):
            x = F.relu(layer(x))
            with torch.no_grad():
                dead_frac = (x == 0).float().mean().item()
                self.activation_stats[name] = {
                    "mean": x.mean().item(),
                    "std":  x.std().item(),
                    "dead_frac": dead_frac      # fraction of zero activations
                }
        return self.fc3(x)

for init in ["zeros", "kaiming"]:
    net = NetWithStats(init).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(10):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(net(xb), yb).backward()
            opt.step()
    # Collect stats on val set
    net.eval()
    with torch.no_grad():
        net(X_v)
        acc = (net(X_v).argmax(1) == y_v).float().mean().item()
    print(f"\nInit={init}  val_acc={acc:.3f}")
    for layer_name, stats in net.activation_stats.items():
        print(f"  {layer_name}: mean={stats['mean']:.4f}  std={stats['std']:.4f}  "
              f"dead_frac={stats['dead_frac']:.2%}")
    if init == "zeros":
        print("  → All neurons learn identically (symmetry problem) — zero init is broken")


## Real-World Example 2 — Small Init for Deep ResNets

In [ ]:
# ── RW2: scaled init (1/sqrt(depth)) for deep residual networks ──────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy, math

torch.manual_seed(42)

X = torch.randn(500, 32)
y = (X[:, 0] - X[:, 1] > 0).long()
loader = DataLoader(TensorDataset(X[:400], y[:400]), batch_size=32, shuffle=True)
X_v, y_v = X[400:].to(device), y[400:].to(device)

N_BLOCKS = 6

class ResBlock(nn.Module):
    def __init__(self, dim, scale=1.0):
        super().__init__()
        self.fc1    = nn.Linear(dim, dim)
        self.fc2    = nn.Linear(dim, dim)
        self.scale  = scale   # multiplier on residual branch
        self._init()

    def _init(self):
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity="relu")
        nn.init.zeros_(self.fc1.bias)
        # Scale output of residual branch to prevent explosion in deep networks
        nn.init.normal_(self.fc2.weight, std=self.scale / math.sqrt(N_BLOCKS))
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        return x + self.fc2(F.relu(self.fc1(x)))   # residual connection

class ResNet(nn.Module):
    def __init__(self, scale=1.0):
        super().__init__()
        self.proj   = nn.Linear(32, 64)
        self.blocks = nn.ModuleList([ResBlock(64, scale) for _ in range(N_BLOCKS)])
        self.fc     = nn.Linear(64, 2)
    def forward(self, x):
        h = F.relu(self.proj(x))
        for b in self.blocks: h = b(h)
        return self.fc(h)

def train_resnet(scale, epochs=30):
    net = ResNet(scale).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    init_loss = None
    for ep in range(epochs):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(net(xb), yb)
            if init_loss is None: init_loss = loss.item()
            loss.backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        acc = (net(X_v).argmax(1) == y_v).float().mean().item()
    return init_loss, acc

print(f"{'Scale':>8}  {'Init Loss':>10}  {'Val Acc':>8}  Note")
print("-" * 50)
for sc in [0.01, 0.1, 0.5, 1.0, 2.0]:
    il, acc = train_resnet(sc)
    note = "too small (slow start)" if sc < 0.05 else            "too large (explodes)"   if sc > 1.5  else "good"
    print(f"{sc:>8.2f}  {il:>10.4f}  {acc:>8.3f}  {note}")
print("\nScale ≈ 1/sqrt(depth) avoids signal explosion while maintaining gradient flow.")


## Real-World Example 3 — Gradient Flow Visualisation per Layer

In [ ]:
# ── RW3: gradient norm per layer for different init schemes ──────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

torch.manual_seed(42)

X = torch.randn(200, 32)
y = (X[:, 0] > 0).long()

def make_net_for_grad_flow(init_scheme):
    net = nn.Sequential(
        nn.Linear(32, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 64),  nn.ReLU(),
        nn.Linear(64, 2)
    )
    linear_layers = [m for m in net if isinstance(m, nn.Linear)]
    for fc in linear_layers:
        if init_scheme == "xavier":
            nn.init.xavier_uniform_(fc.weight)
        elif init_scheme == "kaiming":
            nn.init.kaiming_normal_(fc.weight, nonlinearity="relu")
        elif init_scheme == "constant":
            nn.init.constant_(fc.weight, 0.1)
        nn.init.zeros_(fc.bias)
    return net

def get_gradient_norms(net, X, y):
    # Compute gradient norm at each Linear layer for a single forward-backward pass.
    net = net.to(device)
    X_d, y_d = X.to(device), y.to(device)
    out  = net(X_d)
    loss = F.cross_entropy(out, y_d)
    net.zero_grad()
    loss.backward()
    norms = []
    for m in net:
        if isinstance(m, nn.Linear):
            norms.append(m.weight.grad.norm().item())
    return norms

init_schemes = ["xavier", "kaiming", "constant"]
layer_names  = [f"L{i+1}" for i in range(5)]

print(f"{'Init':<10}  " + "  ".join(f"{n:>10}" for n in layer_names))
print("-" * 65)
for sch in init_schemes:
    net   = make_net_for_grad_flow(sch)
    norms = get_gradient_norms(net, X, y)
    row   = f"{sch:<10}  " + "  ".join(f"{n:>10.6f}" for n in norms)
    print(row)

print("\nKey insight:")
print("  Xavier/Kaiming: gradient norms similar across layers (stable flow)")
print("  Constant init:  symmetry breaking fails, gradients may vanish or explode")
print("  Vanishing: L1 norm >> L5 norm — gradients lost before reaching early layers")
